# Pipeline Huấn luyện & Đánh giá Mô hình ResNet-18 cho Bò (Cow)

Notebook này thực thi toàn bộ quy trình chuẩn hóa từ chuẩn bị dữ liệu, huấn luyện 2 giai đoạn (Transfer Learning), đánh giá trên tập kiểm thử (Test set) và xuất báo cáo kết quả cho bài toán nhận diện bệnh ở **bệnh ở bò**.

---

### Luồng thực thi chuẩn hóa (Execution Flow):
1. **Khởi tạo & Cấu hình**: Thiết lập môi trường, seed ngẫu nhiên (`seed=42`) và xác định trước toàn bộ đường dẫn xuất file chuẩn hóa.
2. **Nạp dữ liệu & DataLoader**: Đọc dataset (`train`, `val`, `test`), tự động tính toán trọng số nghịch đảo tần suất (*Inverse Frequency Class Weights*) để chống mất cân bằng dữ liệu.
3. **Kiểm tra dữ liệu mẫu (Data Sanity Check)**: Kiểm tra tensor batch shape, pixel range và chạy forward pass smoke test.
4. **Phase 1 - Huấn luyện Feature Extraction**: Khóa backbone ResNet-18, chỉ huấn luyện Classification Head mới (10 epoch, lr=1e-3, AdamW + CosineAnnealing).
5. **Phase 1 - Đánh giá**: Nạp checkpoint tốt nhất của Phase 1, đánh giá trên Test set, xuất báo cáo, ma trận nhầm lẫn và biểu đồ học tập.
6. **Phase 2 - Huấn luyện Full Fine-tuning**: Mở khóa toàn bộ backbone, tinh chỉnh toàn bộ trọng số mạng với learning rate nhỏ hơn (15 epoch, lr=1e-4).
7. **Phase 2 - Đánh giá toàn diện**: Nạp checkpoint tốt nhất của Phase 2, đánh giá toàn diện trên Test set và xuất báo cáo kết quả cuối cùng.
8. **Tổng kết hiệu năng**: So sánh đối chiếu chỉ số giữa Phase 1 và Phase 2.

In [ ]:
from pathlib import Path
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Xác định thư mục gốc của project (PROJECT_ROOT)
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

# 2. Định nghĩa loài vật và đường dẫn dữ liệu
ANIMAL = "cow"
DATA_DIR = PROJECT_ROOT / "data" / "Cow_data"

# 3. Import các hàm tiện ích và đường dẫn xuất chuẩn hóa
from src.utils import set_seed, get_pipeline_output_paths, ANIMAL_DISPLAY_NAMES
from src.dataset import load_dataset, make_loaders
from src.model import build_model, count_parameters, unfreeze_backbone, load_checkpoint
from src.train import train_model
from src.evaluate import run_full_evaluation, plot_training_curves

# 4. Cố định seed ngẫu nhiên (seed=42) để đảm bảo tính tái lập kết quả
set_seed(42)

# 5. Khởi tạo và lấy trước các đường dẫn file output chuẩn hóa
paths_p1 = get_pipeline_output_paths(PROJECT_ROOT, animal=ANIMAL, phase="phase1")
paths_p2 = get_pipeline_output_paths(PROJECT_ROOT, animal=ANIMAL, phase="phase2")

# 6. Kiểm tra cấu hình phần cứng
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print(f"PIPELINE HUẤN LUYỆN VÀ ĐÁNH GIÁ: BÒ (COW)")
print("=" * 70)
print(f"Thư mục dự án (PROJECT_ROOT) : {PROJECT_ROOT}")
print(f"Thư mục dữ liệu (DATA_DIR)    : {DATA_DIR}")
print(f"Dữ liệu tồn tại trên máy     : {DATA_DIR.exists()}")
print(f"Phiên bản PyTorch             : {torch.__version__}")
print(f"Thiết bị tính toán (Device)   : {device}")
if device.type == "cuda":
    print(f"Tên GPU                       : {torch.cuda.get_device_name(0)}")
print("=" * 70)

In [ ]:
# 1. Nạp dataset qua ImageFolder với 3 tập: train, val, test
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục dữ liệu tại: {DATA_DIR}\n"
        "Vui lòng giải nén bộ dữ liệu vào đúng đường dẫn trên trước khi tiếp tục."
    )

train_dataset, val_dataset, test_dataset, info = load_dataset(DATA_DIR)

# 2. Khởi tạo DataLoaders với batch_size=32 (phù hợp GPU 4GB VRAM trở lên)
# Cấu hình num_workers tối ưu và an toàn trên Windows
num_workers = 4 if torch.cuda.is_available() else 0
train_loader, val_loader, test_loader = make_loaders(
    train_dataset,
    val_dataset,
    test_dataset,
    batch_size=32,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(num_workers > 0),
)

print("\n--- THÔNG TIN TẬP DỮ LIỆU ---")
print(f"Kích thước split        : Train={info['split_sizes']['train']:,} | Val={info['split_sizes']['val']:,} | Test={info['split_sizes']['test']:,} | Tổng={info['split_sizes']['total']:,}")
print(f"Số lượng lớp bệnh       : {info['n_classes']}")
print("Danh sách các lớp bệnh  :")
for idx, c_name in enumerate(info['class_names']):
    print(f"  [{idx}] {c_name}")
print(f"\nTrọng số phân lớp (Class Weights):\n{info['class_weights']}")

In [ ]:
# 1. Trích xuất thử 1 mini-batch từ DataLoader để kiểm tra kích thước và giá trị pixel
images, labels = next(iter(train_loader))

print("--- KIỂM TRA BATCH ĐẦU VÀO ---")
print(f"Kích thước tensor ảnh (B, C, H, W) : {images.shape}")
print(f"Kích thước tensor nhãn (B,)       : {labels.shape}")
print(f"Dải giá trị pixel sau normalize    : Min = {images.min().item():.3f}, Max = {images.max().item():.3f}")
print(f"Nhãn 10 mẫu đầu tiên               : {labels[:10].tolist()}")

# 2. Khởi tạo mô hình ResNet-18 cho Phase 1 (khóa backbone, chỉ học classification head)
model = build_model(
    n_classes=info["n_classes"],
    freeze_backbone=True,
).to(device)

params_p1 = count_parameters(model)
print("\n--- THỐNG KÊ THAM SỐ MÔ HÌNH (PHASE 1) ---")
print(f"Tổng số tham số     : {params_p1['total']:,}")
print(f"Tham số huấn luyện  : {params_p1['trainable']:,} (Chỉ lớp phân loại cuối)")
print(f"Tham số bị đóng băng: {params_p1['frozen']:,}")

# 3. Chạy thử nghiệm forward pass (Smoke Test)
with torch.no_grad():
    sample_outputs = model(images.to(device))
print(f"\nForward smoke test thành công! Shape output: {sample_outputs.shape}")

In [ ]:
# Huấn luyện Phase 1 (Feature Extraction):
# - Đóng băng (Freeze) backbone ResNet-18 pretrained ImageNet
# - Tốc độ học: lr=1e-3, Optimizer: AdamW, Scheduler: CosineAnnealingLR
# - Hàm mất mát: CrossEntropyLoss kèm Class Weights chống lệch lớp
# - Tự động lưu checkpoint xuất sắc nhất theo Validation Accuracy

print("Bắt đầu huấn luyện Phase 1 (10 epoch, lr=0.001, khóa backbone)...")

phase1_history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=10,
    learning_rate=0.001,
    class_weights=info["class_weights"].to(device),
    device=device,
    checkpoint_path=paths_p1["model_checkpoint"],
    history_path=paths_p1["history_json"],
    phase_name=f"phase1_frozen_backbone_{ANIMAL}",
    class_names=info["class_names"],
)

print(f"\nHoàn tất huấn luyện Phase 1! Best Val Acc: {phase1_history['best_val_accuracy']:.4f}")
print(f"Checkpoint tốt nhất đã lưu tại : {paths_p1['model_checkpoint'].name}")
print(f"Lịch sử huấn luyện đã lưu tại  : {paths_p1['history_json'].name}")

In [ ]:
# QUAN TRỌNG: Nạp lại checkpoint có Validation Accuracy tốt nhất trước khi đánh giá Test
print(f"Nạp lại checkpoint Phase 1 tốt nhất từ: {paths_p1['model_checkpoint'].name}")
load_checkpoint(model, paths_p1["model_checkpoint"], device=device)

# Lấy danh sách tên hiển thị tiếng Việt nếu có
display_names = ANIMAL_DISPLAY_NAMES.get(ANIMAL, info["class_names"])

# Thực hiện đánh giá toàn diện trên Test set
phase1_report = run_full_evaluation(
    model=model,
    test_loader=test_loader,
    class_names=info["class_names"],
    device=device,
    results_dir=PROJECT_ROOT / "results",
    figures_dir=PROJECT_ROOT / "figures",
    label="phase1",
    display_names=display_names,
    prefix=f"{ANIMAL}_",
)

# Vẽ và lưu biểu đồ Loss & Accuracy qua các epoch của Phase 1
plot_training_curves(
    phase1_history,
    save_path=paths_p1["training_curves_png"],
)

print(f"\nĐã lưu báo cáo phân loại Phase 1 : {paths_p1['report_json'].name} & .csv")
print(f"Đã lưu ma trận nhầm lẫn          : {paths_p1['confusion_matrix_png'].name}")
print(f"Đã lưu biểu đồ F1 từng lớp       : {paths_p1['per_class_f1_png'].name}")
print(f"Đã lưu đồ thị quá trình học      : {paths_p1['training_curves_png'].name}")

In [ ]:
# Mở khóa toàn bộ tham số của mạng ResNet-18 để tinh chỉnh ở Phase 2
unfreeze_backbone(model)

params_p2 = count_parameters(model)
print("--- THỐNG KÊ THAM SỐ MÔ HÌNH (PHASE 2 - FINE-TUNING) ---")
print(f"Tổng số tham số     : {params_p2['total']:,}")
print(f"Tham số huấn luyện  : {params_p2['trainable']:,} (Toàn bộ tham số đã mở khóa)")
print(f"Tham số bị đóng băng: {params_p2['frozen']:,}")

In [ ]:
# Huấn luyện Phase 2 (Full Fine-tuning):
# - Mở khóa toàn bộ backbone và classification head
# - Tốc độ học nhỏ hơn 10 lần (lr=1e-4) để tinh chỉnh mượt mà, không làm mất đặc trưng ImageNet
# - Tự động lưu checkpoint có Validation Accuracy tốt nhất

print("Bắt đầu huấn luyện Phase 2 (15 epoch, lr=0.0001, fine-tune toàn bộ model)...")

phase2_history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=15,
    learning_rate=0.0001,
    class_weights=info["class_weights"].to(device),
    device=device,
    checkpoint_path=paths_p2["model_checkpoint"],
    history_path=paths_p2["history_json"],
    phase_name=f"phase2_full_finetuning_{ANIMAL}",
    class_names=info["class_names"],
)

print(f"\nHoàn tất huấn luyện Phase 2! Best Val Acc: {phase2_history['best_val_accuracy']:.4f}")
print(f"Checkpoint tốt nhất đã lưu tại : {paths_p2['model_checkpoint'].name}")
print(f"Lịch sử huấn luyện đã lưu tại  : {paths_p2['history_json'].name}")

In [ ]:
# QUAN TRỌNG: Nạp lại checkpoint có Validation Accuracy tốt nhất của Phase 2
print(f"Nạp lại checkpoint Phase 2 tốt nhất từ: {paths_p2['model_checkpoint'].name}")
load_checkpoint(model, paths_p2["model_checkpoint"], device=device)

# Thực hiện đánh giá toàn diện trên Test set
phase2_report = run_full_evaluation(
    model=model,
    test_loader=test_loader,
    class_names=info["class_names"],
    device=device,
    results_dir=PROJECT_ROOT / "results",
    figures_dir=PROJECT_ROOT / "figures",
    label="phase2",
    display_names=display_names,
    prefix=f"{ANIMAL}_",
)

# Vẽ và lưu biểu đồ Loss & Accuracy qua các epoch của Phase 2
plot_training_curves(
    phase2_history,
    save_path=paths_p2["training_curves_png"],
)

print(f"\nĐã lưu báo cáo phân loại Phase 2 : {paths_p2['report_json'].name} & .csv")
print(f"Đã lưu ma trận nhầm lẫn          : {paths_p2['confusion_matrix_png'].name}")
print(f"Đã lưu biểu đồ F1 từng lớp       : {paths_p2['per_class_f1_png'].name}")
print(f"Đã lưu đồ thị quá trình học      : {paths_p2['training_curves_png'].name}")

In [ ]:
# Tổng hợp và so sánh số liệu giữa Phase 1 và Phase 2
summary_data = {
    "Chỉ số": ["Test Accuracy", "Macro F1", "Weighted F1", "Best Val Accuracy", "Learning Rate", "Epochs"],
    "Phase 1 (Frozen Backbone)": [
        f"{phase1_report['overall_accuracy'] * 100:.2f}%",
        f"{phase1_report['macro avg']['f1-score'] * 100:.2f}%",
        f"{phase1_report['weighted avg']['f1-score'] * 100:.2f}%",
        f"{phase1_history['best_val_accuracy'] * 100:.2f}%",
        "0.001 (1e-3)",
        "10",
    ],
    "Phase 2 (Full Fine-tuning)": [
        f"{phase2_report['overall_accuracy'] * 100:.2f}%",
        f"{phase2_report['macro avg']['f1-score'] * 100:.2f}%",
        f"{phase2_report['weighted avg']['f1-score'] * 100:.2f}%",
        f"{phase2_history['best_val_accuracy'] * 100:.2f}%",
        "0.0001 (1e-4)",
        "15",
    ],
}

summary_df = pd.DataFrame(summary_data)
print("\n" + "=" * 70)
print(f"TỔNG KẾT SO SÁNH HIỆU NĂNG: {ANIMAL.upper()}")
print("=" * 70)
print(summary_df.to_string(index=False))
print("=" * 70)